# OpenSearch Serverless: Hybrid Search & Benchmark

이 노트북에서는 **Amazon OpenSearch Serverless**에 비디오 임베딩을 저장하고 검색합니다.

OpenSearch Serverless의 핵심 차별점은 **하이브리드 검색**입니다. 벡터 유사도 검색(시맨틱)과 키워드 검색(BM25)을 동시에 수행할 수 있어, 시각 정보와 텍스트 정보가 혼합된 쿼리에 효과적입니다.

이 노트북에서 다루는 내용:
1. 벡터 인제스트 및 속도 측정
2. k-NN 검색 vs 하이브리드 검색 비교
3. 검색 레이턴시 벤치마크 (p50/p95/p99)
4. 결과를 `benchmark_opensearch.json`에 저장 → 04에서 S3 Vectors와 비교

**사전 조건:** `01_setup_and_embeddings.ipynb` 실행 완료

## 0. Setup

In [ ]:
%pip install -r requirements.txt -Uq

In [ ]:
import json, time
import boto3
from opensearchpy import OpenSearch, RequestsHttpConnection
from requests_aws4auth import AWS4Auth

with open("config.json") as f:
    config = json.load(f)

REGION = config["REGION"]
S3_BUCKET = config["S3_BUCKET"]
EMBEDDINGS_S3_PREFIX = config["EMBEDDINGS_S3_PREFIX"]
EMBEDDING_DIMENSION = config["EMBEDDING_DIMENSION"]
VIDEO_IDS = config["VIDEO_IDS"]
OPENSEARCH_ENDPOINT = config.get("OPENSEARCH_ENDPOINT", "")

if not OPENSEARCH_ENDPOINT:
    raise ValueError("OPENSEARCH_ENDPOINT가 config.json에 없습니다. 01 노트북에서 설정해주세요.")

session = boto3.Session()
s3 = session.client("s3", region_name=REGION)

print(f"✅ OpenSearch: {OPENSEARCH_ENDPOINT}")
print(f"   벡터 차원: {EMBEDDING_DIMENSION}, 비디오: {len(VIDEO_IDS)}개")

### ⚙️ Parameters

| 파라미터 | 설명 | 권장 범위 |
|---------|------|----------|
| `INDEX_NAME` | 벡터 인덱스 이름 (CloudFormation 데이터 접근 정책과 일치해야 함) | `video-embeddings-index` |
| `INGEST_BATCH_SIZE` | bulk API 호출당 문서 수. 클수록 API 호출 횟수 감소 | 50-500 |
| `SEARCH_K` | k-NN 검색에서 반환할 최근접 이웃 수 | 5, 10, 20 |
| `HYBRID_VECTOR_WEIGHT` | 하이브리드 검색에서 벡터 유사도 가중치 | 0.5-0.9 |
| `HYBRID_TEXT_WEIGHT` | 하이브리드 검색에서 키워드 매칭 가중치 | 0.1-0.5 |

In [ ]:
INDEX_NAME = "video-embeddings-index"
INGEST_BATCH_SIZE = 200
SEARCH_K = 5
HYBRID_VECTOR_WEIGHT = 0.7
HYBRID_TEXT_WEIGHT = 0.3

## 1. Connect to OpenSearch Serverless

CloudFormation으로 프로비저닝된 OpenSearch Serverless 컬렉션에 연결합니다. SigV4 인증을 사용합니다.

In [ ]:
host = OPENSEARCH_ENDPOINT.replace("https://", "")
creds = session.get_credentials().get_frozen_credentials()
auth = AWS4Auth(creds.access_key, creds.secret_key, REGION, "aoss", session_token=creds.token)

os_client = OpenSearch(
    hosts=[{"host": host, "port": 443}],
    http_auth=auth, use_ssl=True, verify_certs=True,
    connection_class=RequestsHttpConnection, timeout=60
)
print(f"✅ 연결 완료: {host}")

## 2. Ingest Embeddings

S3에서 임베딩을 로드하고 OpenSearch에 인제스트합니다. 인제스트 시간을 측정하여 04에서 S3 Vectors와 비교합니다.

> 📖 **OpenSearch bulk API**: NDJSON 형식으로 여러 문서를 한 번에 인덱싱합니다. 배치 크기는 문서 수가 아닌 요청 크기(기본 100MB) 기반으로 제한되므로, 배치를 크게 잡을 수 있습니다.

In [ ]:
def load_embeddings_from_s3():
    """Load embeddings from S3 (searches multiple paths, handles Workshop 1 and 4 formats)."""
    vectors, found_files = [], []
    for prefix in [EMBEDDINGS_S3_PREFIX, 'embeddings/videos/', 'embeddings/']:
        resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=prefix, MaxKeys=200)
        for obj in resp.get('Contents', []):
            if obj['Key'].endswith('.json') and '/config.json' not in obj['Key']:
                found_files.append(obj['Key'])
        if found_files: break

    # 통합 파일({video_id}.json)이 있으면 output.json/manifest.json 스킵 (중복 방지)
    consolidated = [f for f in found_files if f.split('/')[-1] not in ('output.json', 'manifest.json')]
    if consolidated:
        found_files = consolidated

    for s3_key in found_files:
        try: data = json.loads(s3.get_object(Bucket=S3_BUCKET, Key=s3_key)['Body'].read().decode())
        except: continue
        segs = data.get('vectors', data.get('data', []))
        vid = data.get('video_id', '')
        if not vid:
            basename = s3_key.split('/')[-1].replace('.json', '')
            if basename == 'output':
                # Try to get video filename from Bedrock output metadata
                import os as _os
                s3_uri = data.get('inputVideo', {}).get('s3Uri', '')
                if not s3_uri:
                    s3_uri = data.get('video', {}).get('mediaSource', {}).get('s3Location', {}).get('uri', '')
                if s3_uri:
                    vid = _os.path.splitext(_os.path.basename(s3_uri))[0]
                else:
                    vid = s3_key.split('/')[-3]
            else:
                vid = basename
        for seg in segs:
            if 'embedding' not in seg: continue
            vectors.append({
                'video_id': vid, 'embedding': seg['embedding'],
                'start_sec': seg.get('startSec', seg.get('start_time', 0)),
                'end_sec': seg.get('endSec', seg.get('end_time', 0)),
                'scope': seg.get('embeddingScope', seg.get('embedding_scope', 'clip')),
                'description': f"{vid} video segment"
            })
    return vectors

vectors = load_embeddings_from_s3()
print(f"✅ S3에서 {len(vectors)}개 벡터 로드 완료")

In [ ]:
# 기존 인덱스 확인 → 삭제 후 재생성 (깨끗한 벤치마크를 위해)
try:
    if os_client.indices.exists(index=INDEX_NAME):
        os_client.indices.delete(index=INDEX_NAME)
        print(f"   🗑️  기존 인덱스 삭제")
        time.sleep(3)
except: pass

os_client.indices.create(index=INDEX_NAME, body={
    "settings": {"index": {"knn": True}},
    "mappings": {"properties": {
        "embedding": {"type": "knn_vector", "dimension": EMBEDDING_DIMENSION,
                      "method": {"engine": "faiss", "name": "hnsw", "space_type": "cosinesimil"}},
        "video_id": {"type": "keyword"}, "start_sec": {"type": "float"},
        "end_sec": {"type": "float"}, "scope": {"type": "keyword"}, "description": {"type": "text"}
    }}
})
print(f"✅ 인덱스 '{INDEX_NAME}' 생성 (faiss/hnsw/cosinesimil)")
time.sleep(10)

# Ingest with timing
print(f"   인제스트 중: {len(vectors)}개 벡터, batch_size={INGEST_BATCH_SIZE}...")
ingest_start = time.time()
ingested = 0
for b in range(0, len(vectors), INGEST_BATCH_SIZE):
    batch = vectors[b:b + INGEST_BATCH_SIZE]
    bulk_body = []
    for v in batch:
        bulk_body.append(json.dumps({"index": {"_index": INDEX_NAME}}))
        bulk_body.append(json.dumps(v))
    resp = os_client.bulk(body="\n".join(bulk_body) + "\n")
    if resp.get('errors'):
        for item in resp['items']:
            if 'error' in item.get('index', {}):
                print(f"   ❌ {item['index']['error']}")
                break
    else:
        ingested += len(batch)
ingest_time = time.time() - ingest_start

# OpenSearch Serverless는 자동 refresh 간격이 있어 인덱싱 반영까지 대기 필요
print(f"   {ingested}개 전송 완료 ({ingest_time:.2f}초), 인덱싱 반영 대기 중...")
for _ in range(6):
    time.sleep(10)
    count = os_client.count(index=INDEX_NAME)["count"]
    if count > 0:
        break
print(f"✅ 인제스트 완료: {count}개 인덱싱 확인, 인제스트 시간 {ingest_time:.2f}초")

## 3. Hybrid Search

OpenSearch의 핵심 기능인 **하이브리드 검색**을 실습합니다.

| 검색 방식 | 작동 원리 | 강점 |
|----------|----------|------|
| **k-NN (벡터)** | 코사인 유사도로 의미적으로 가까운 벡터를 찾음 | 시각적/개념적 유사성 |
| **BM25 (키워드)** | 쿼리 단어가 문서에 얼마나 자주 등장하는지 계산 | 정확한 키워드 매칭 |
| **하이브리드** | 두 스코어를 정규화 후 가중 평균으로 결합 | 두 방식의 장점 결합 |

OpenSearch는 `search pipeline`의 `normalization-processor`로 서로 다른 스코어 체계를 0-1 범위로 정규화한 뒤 결합합니다. 이것이 단순 `bool` 쿼리보다 정확한 랭킹을 제공하는 이유입니다.

> 💡 **S3 Vectors와의 차이**: S3 Vectors는 벡터 유사도 검색만 지원합니다. 키워드 검색이 필요한 경우 OpenSearch가 유일한 선택입니다.

In [ ]:
# Search pipeline 생성
pipeline_body = {
    "description": "Normalization pipeline for hybrid search",
    "phase_results_processors": [{
        "normalization-processor": {
            "normalization": {"technique": "min_max"},
            "combination": {"technique": "arithmetic_mean",
                            "parameters": {"weights": [HYBRID_VECTOR_WEIGHT, HYBRID_TEXT_WEIGHT]}}
        }
    }]
}
os_client.http.put("/_search/pipeline/hybrid-pipeline", body=pipeline_body)
print(f"✅ Search pipeline 생성 (vector={HYBRID_VECTOR_WEIGHT}, text={HYBRID_TEXT_WEIGHT})")

In [ ]:
# Text-to-embedding
from botocore.config import Config
bedrock = session.client("bedrock-runtime", region_name=REGION, config=Config(signature_version="v4"))

def embed_text(query):
    resp = bedrock.invoke_model(
        modelId="us.twelvelabs.marengo-embed-3-0-v1:0",
        body=json.dumps({"inputType": "text", "text": {"inputText": query}})
    )
    return json.loads(resp["body"].read())["data"][0]["embedding"]

print("✅ 텍스트 임베딩 함수 준비 완료")

In [ ]:
# ==============================
# 🔎 검색 쿼리를 입력하세요
# ==============================
SEARCH_QUERY = "goal scene in a soccer match"  # ← 변경해보세요!

query_vector = embed_text(SEARCH_QUERY)
print(f"✅ 쿼리: '{SEARCH_QUERY}' → {len(query_vector)}차원 벡터")

In [ ]:
def fmt_hit(hit):
    src = hit['_source']
    start = src.get('start_sec', src.get('start_time', 0))
    end = src.get('end_sec', src.get('end_time', 0))
    scope = src.get('scope', src.get('embedding_scope', '?'))
    return f"score={hit['_score']:.4f} | {src.get('video_id','?')} | {start:.1f}s-{end:.1f}s | {scope}"

# 1) k-NN 검색
t0 = time.time()
knn_resp = os_client.search(index=INDEX_NAME, body={"size": SEARCH_K, "query": {"knn": {"embedding": {"vector": query_vector, "k": SEARCH_K}}}})
knn_latency = (time.time() - t0) * 1000

print(f"🔍 k-NN 검색 (latency={knn_latency:.1f}ms):")
for i, hit in enumerate(knn_resp["hits"]["hits"], 1):
    print(f"  {i}. {fmt_hit(hit)}")

# 2) 하이브리드 검색
hybrid_body = {"size": SEARCH_K, "query": {"hybrid": {"queries": [
    {"knn": {"embedding": {"vector": query_vector, "k": SEARCH_K}}},
    {"match": {"description": SEARCH_QUERY}}
]}}}
t0 = time.time()
hybrid_resp = os_client.search(index=INDEX_NAME, body=hybrid_body, params={"search_pipeline": "hybrid-pipeline"})
hybrid_latency = (time.time() - t0) * 1000

print(f"\n🔍 하이브리드 검색 (latency={hybrid_latency:.1f}ms):")
for i, hit in enumerate(hybrid_resp["hits"]["hits"], 1):
    print(f"  {i}. {fmt_hit(hit)}")

In [ ]:
# 비디오 재생 — k-NN vs 하이브리드 비교
from IPython.display import display, HTML

def build_video_mapping():
    mapping = {}
    for prefix in ['videos/', 'vectordb-workshop/videos/']:
        resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=prefix, MaxKeys=200)
        for obj in resp.get('Contents', []):
            if obj['Key'].endswith('.mp4'):
                import os; mapping[os.path.splitext(os.path.basename(obj['Key']))[0]] = obj['Key']
    for prefix in ['embeddings/videos/', 'vectordb-workshop/embeddings/']:
        resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=prefix, MaxKeys=500)
        for obj in resp.get('Contents', []):
            if obj['Key'].endswith('output.json'):
                try:
                    data = json.loads(s3.get_object(Bucket=S3_BUCKET, Key=obj['Key'])['Body'].read().decode())
                    s3_uri = data.get('inputVideo', {}).get('s3Uri', '')
                    if s3_uri:
                        parts = obj['Key'].split('/')
                        mapping[parts[2] if len(parts) > 2 else parts[-2]] = s3_uri.replace(f's3://{S3_BUCKET}/', '')
                except: continue
    return mapping

video_mapping = build_video_mapping()
print(f"✅ 비디오 매핑: {len(video_mapping)}개")
print(f"   매핑된 ID: {list(video_mapping.keys())}")

def play_result(hit_source, label=''):
    vid = hit_source.get('video_id', '')
    start = hit_source.get('start_sec', hit_source.get('start_time', 0))
    key = video_mapping.get(vid)
    if key:
        url = s3.generate_presigned_url('get_object', Params={'Bucket': S3_BUCKET, 'Key': key}, ExpiresIn=3600)
        display(HTML(f'<h4>{label} ▶️ {vid} ({start:.1f}s~)</h4><video width="480" controls><source src="{url}#t={start}" type="video/mp4"></video>'))
    else:
        print(f"⚠️  {label} 비디오 미발견: '{vid}' (매핑에 없음)")

# k-NN Top 1 vs 하이브리드 Top 1 비교 재생
print('\n📊 k-NN vs 하이브리드 검색 결과 비교:\n')
if knn_resp['hits']['hits']:
    play_result(knn_resp['hits']['hits'][0]['_source'], label='[k-NN #1]')
if hybrid_resp['hits']['hits']:
    play_result(hybrid_resp['hits']['hits'][0]['_source'], label='[하이브리드 #1]')

## 4. Search Benchmark

검색 레이턴시를 체계적으로 측정합니다. 결과는 `benchmark_opensearch.json`에 저장하여 04에서 S3 Vectors와 비교합니다.

### 퍼센타일 레이턴시란?

평균값은 극단적인 값(outlier)에 의해 왜곡될 수 있어, 실제 서비스 품질을 측정할 때는 **퍼센타일**을 사용합니다:

| 지표 | 의미 | 활용 |
|------|------|------|
| **p50** (중앙값) | 요청의 50%가 이 시간 이내에 완료 | 일반적인 사용자 경험 |
| **p95** | 요청의 95%가 이 시간 이내에 완료 | 대부분의 사용자 경험 |
| **p99** | 요청의 99%가 이 시간 이내에 완료 | 최악의 경우 (SLA 기준) |

In [ ]:
BENCHMARK_ITERATIONS = 20
BENCHMARK_QUERY_COUNT = 5

In [ ]:
import numpy as np

query_vectors = [v["embedding"] for v in vectors if v["scope"] == "asset"][:BENCHMARK_QUERY_COUNT]
if not query_vectors: query_vectors = [vectors[0]["embedding"]]

print(f"벤치마크: {len(query_vectors)}개 쿼리 × {BENCHMARK_ITERATIONS}회 반복\n")

benchmark_results = {"service": "OpenSearch Serverless", "vector_count": len(vectors),
                     "ingest_time": ingest_time, "ingest_batch_size": INGEST_BATCH_SIZE, "search": {}}

for k_val in [5, 10, 20]:
    latencies = []
    for _ in range(BENCHMARK_ITERATIONS):
        for qv in query_vectors:
            t0 = time.time()
            os_client.search(index=INDEX_NAME, body={"size": k_val, "query": {"knn": {"embedding": {"vector": qv, "k": k_val}}}})
            latencies.append((time.time() - t0) * 1000)
    latencies.sort()
    n = len(latencies)
    stats = {"p50": latencies[n//2], "p95": latencies[int(n*0.95)], "p99": latencies[int(n*0.99)], "avg": np.mean(latencies)}
    benchmark_results["search"][f"k={k_val}"] = stats
    print(f"  k={k_val:<3} | p50={stats['p50']:.1f}ms | p95={stats['p95']:.1f}ms | p99={stats['p99']:.1f}ms | avg={stats['avg']:.1f}ms")

print("\n✅ 벤치마크 완료")

In [ ]:
# 결과 저장 (04_comparison.ipynb에서 사용)
with open("benchmark_opensearch.json", "w") as f:
    json.dump(benchmark_results, f, indent=2)

print(f"✅ benchmark_opensearch.json 저장 완료")
print(f"   인제스트: {ingest_time:.2f}초 ({len(vectors)}개 벡터, batch={INGEST_BATCH_SIZE})")
print(f"   검색 p50 (k=5): {benchmark_results['search']['k=5']['p50']:.1f}ms")
print(f"\n   다음 단계: 03_s3_vectors.ipynb 실행 후 04_comparison.ipynb에서 비교")